# Video Game Sales

Analise exploratoria orientada por perguntas usando o dataset `gregorut/videogamesales`. O objetivo deste notebook e responder pelo menos 6 perguntas com apoio de graficos e pequenas interpretacoes.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
# Baixa e carrega o dataset
path = kagglehub.dataset_download("gregorut/videogamesales")
csv_files = sorted(file for file in os.listdir(path) if file.endswith(".csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi encontrado no dataset baixado.")

csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

# Remove registros com ano ou publicadora ausentes para estabilizar a analise
df_clean = df.dropna(subset=["Year", "Publisher"]).copy()
df_clean["Year"] = df_clean["Year"].astype(int)

region_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
region_names = {
    "NA_Sales": "America do Norte",
    "EU_Sales": "Europa",
    "JP_Sales": "Japao",
    "Other_Sales": "Outras regioes",
}

print("CSV selecionado:", csv_path)
print("Dimensao original:", df.shape)
print("Dimensao apos limpeza:", df_clean.shape)

df.head()

## Resumo inicial

Antes das perguntas, vale registrar o tamanho da base, o recorte temporal e o total de vendas acumuladas.

In [ ]:
summary = pd.DataFrame(
    {
        "Indicador": [
            "Linhas na base original",
            "Colunas na base original",
            "Linhas apos limpeza",
            "Intervalo de anos",
            "Generos unicos",
            "Plataformas unicas",
            "Publicadoras unicas",
            "Vendas globais somadas",
        ],
        "Valor": [
            len(df),
            df.shape[1],
            len(df_clean),
            f"{df_clean['Year'].min()} - {df_clean['Year'].max()}",
            df_clean['Genre'].nunique(),
            df_clean['Platform'].nunique(),
            df_clean['Publisher'].nunique(),
            round(df_clean['Global_Sales'].sum(), 2),
        ],
    }
)

summary

## Pergunta 1. Onde estao os principais dados ausentes?

A primeira checagem serve para entender se existe alguma coluna que pode comprometer a leitura da base.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

plt.figure(figsize=(10, 4))
sns.barplot(x=missing_counts.values, y=missing_counts.index, hue=missing_counts.index, legend=False, palette="crest")
plt.title("Valores ausentes por coluna")
plt.xlabel("Quantidade de valores ausentes")
plt.ylabel("Coluna")
plt.tight_layout()
plt.show()

missing_counts.to_frame("Valores ausentes")

**Resposta curta:** os problemas de completude estao concentrados em `Year` e `Publisher`. As demais colunas estao completas na base analisada.

## Pergunta 2. Em quais anos as vendas globais atingiram o pico?

Agora observamos a evolucao do mercado ao longo do tempo para localizar os anos mais fortes.

In [ ]:
yearly_sales = df_clean.groupby("Year", as_index=False)["Global_Sales"].sum()
peak_sales_row = yearly_sales.loc[yearly_sales["Global_Sales"].idxmax()]

plt.figure(figsize=(13, 6))
sns.lineplot(data=yearly_sales, x="Year", y="Global_Sales", marker="o", linewidth=2.5, color="#0f766e")
plt.scatter(peak_sales_row["Year"], peak_sales_row["Global_Sales"], color="#d97706", s=120, zorder=5)
plt.text(peak_sales_row["Year"] + 0.25, peak_sales_row["Global_Sales"] + 10, f"Pico: {int(peak_sales_row['Year'])}")
plt.title("Vendas globais por ano")
plt.xlabel("Ano")
plt.ylabel("Vendas globais (milhoes)")
plt.tight_layout()
plt.show()

yearly_sales.sort_values("Global_Sales", ascending=False).head(10)

**Resposta curta:** o maior pico de vendas ocorre em **2008**, seguido de perto por **2009** e **2010**. O fim dos anos 2000 concentra o momento mais forte do mercado dentro do dataset.

## Pergunta 3. Em quais anos houve mais lancamentos de jogos?

Nem sempre vender mais significa ter mais titulos, entao vale separar volume de lancamentos e volume de vendas.

In [ ]:
games_released_per_year = df_clean.groupby("Year", as_index=False).size().rename(columns={"size": "Games_Released"})
peak_release_row = games_released_per_year.loc[games_released_per_year["Games_Released"].idxmax()]

plt.figure(figsize=(13, 6))
sns.lineplot(data=games_released_per_year, x="Year", y="Games_Released", linewidth=2.5, color="#1d4ed8")
plt.fill_between(games_released_per_year["Year"], games_released_per_year["Games_Released"], alpha=0.20, color="#93c5fd")
plt.scatter(peak_release_row["Year"], peak_release_row["Games_Released"], color="#e76f51", s=120, zorder=5)
plt.text(peak_release_row["Year"] + 0.25, peak_release_row["Games_Released"] + 25, f"Pico: {int(peak_release_row['Year'])}")
plt.title("Quantidade de jogos lancados por ano")
plt.xlabel("Ano")
plt.ylabel("Quantidade de jogos")
plt.tight_layout()
plt.show()

games_released_per_year.sort_values("Games_Released", ascending=False).head(10)

**Resposta curta:** o maior volume de lancamentos aparece em **2009**, com mais de **1.400 jogos**. Isso reforca que o periodo de 2007 a 2011 foi muito intenso tanto em oferta quanto em vendas.

## Pergunta 4. Quais generos mais venderam no mercado global?

Aqui o foco deixa de ser o tempo e passa a ser o tipo de experiencia mais forte em vendas acumuladas.

In [ ]:
top_genres = (
    df_clean.groupby("Genre", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
    .head(10)
)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_genres, x="Global_Sales", y="Genre", hue="Genre", legend=False, palette="viridis")
plt.title("Top 10 generos por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Genero")
plt.tight_layout()
plt.show()

top_genres

**Resposta curta:** `Action` lidera com folga, seguido por `Sports` e `Shooter`. Isso sugere um mercado historicamente muito concentrado em jogos de alto apelo popular e grande escala comercial.

## Pergunta 5. Quais plataformas dominaram as vendas globais?

Depois dos generos, o passo seguinte e olhar para os consoles e sistemas que mais capturaram vendas.

In [ ]:
top_platforms = (
    df_clean.groupby("Platform", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
    .head(10)
)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_platforms, x="Global_Sales", y="Platform", hue="Platform", legend=False, palette="mako")
plt.title("Top 10 plataformas por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Plataforma")
plt.tight_layout()
plt.show()

top_platforms

**Resposta curta:** a `PS2` lidera o ranking historico, com `X360`, `PS3` e `Wii` logo atras. A geracao dos anos 2000 concentra boa parte do valor acumulado da base.

## Pergunta 6. Quais publicadoras concentraram mais vendas?

Essa pergunta ajuda a entender quem capturou mais resultado comercial ao longo do periodo.

In [ ]:
top_publishers = (
    df_clean.groupby("Publisher", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=False)
    .head(10)
)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_publishers, x="Global_Sales", y="Publisher", hue="Publisher", legend=False, palette="rocket")
plt.title("Top 10 publicadoras por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Publicadora")
plt.tight_layout()
plt.show()

top_publishers

**Resposta curta:** `Nintendo` aparece muito acima das demais, enquanto `Electronic Arts` e `Activision` formam o segundo bloco de destaque. A lideranca da Nintendo e coerente com a forca de seus jogos mais vendidos.

## Pergunta 7. Quais jogos foram os maiores campeoes de vendas?

Depois do nivel agregado, vale olhar quais titulos puxam o topo do ranking global.

In [ ]:
top_games = df_clean.nlargest(10, "Global_Sales")[["Name", "Platform", "Global_Sales"]].copy()
top_games["Game"] = top_games["Name"] + " (" + top_games["Platform"] + ")"
top_games = top_games.sort_values("Global_Sales", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_games, x="Global_Sales", y="Game", hue="Game", legend=False, palette="flare")
plt.title("Top 10 jogos por vendas globais")
plt.xlabel("Vendas globais (milhoes)")
plt.ylabel("Jogo")
plt.tight_layout()
plt.show()

top_games[["Game", "Global_Sales"]].sort_values("Global_Sales", ascending=False)

**Resposta curta:** `Wii Sports` lidera com muita distancia do restante. O topo tambem mostra uma presenca forte da Nintendo, especialmente em franquias e consoles muito populares.

## Pergunta 8. Como as vendas se distribuem entre as regioes?

O total global esconde perfis regionais diferentes, entao esta pergunta compara o peso de cada mercado.

In [ ]:
regional_totals = df_clean[region_cols].sum().sort_values(ascending=False)
regional_labels = [region_names[col] for col in regional_totals.index]
regional_colors = sns.color_palette("Set2", n_colors=4)

plt.figure(figsize=(8, 8))
plt.pie(
    regional_totals.values,
    labels=regional_labels,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.82,
    colors=regional_colors,
    wedgeprops={"width": 0.45, "edgecolor": "white"},
)
plt.title("Participacao regional nas vendas globais")
plt.tight_layout()
plt.show()

pd.DataFrame({"Regiao": regional_labels, "Vendas": regional_totals.values})

**Resposta curta:** a **America do Norte** lidera com bastante folga, seguida por **Europa** e **Japao**. Isso mostra que o mercado global da base e fortemente puxado pelo Ocidente.

## Pergunta 9. O peso das regioes muda entre os generos lideres?

Por fim, observamos se a distribuicao regional muda quando olhamos apenas para os generos que mais vendem.

In [ ]:
genre_region = df_clean.groupby("Genre")[region_cols].sum()
top_genres_region = genre_region.loc[top_genres["Genre"].head(5)]

plt.figure(figsize=(12, 6))
top_genres_region.plot(kind="bar", stacked=True, colormap="viridis")
plt.title("Distribuicao regional nos 5 generos lideres")
plt.xlabel("Genero")
plt.ylabel("Vendas (milhoes)")
plt.xticks(rotation=0)
plt.legend(title="Regiao", labels=[region_names[col] for col in region_cols])
plt.tight_layout()
plt.show()

top_genres_region.rename(columns=region_names)

**Resposta curta:** sim. `Action`, `Sports` e `Shooter` dependem mais de America do Norte e Europa, enquanto `Role-Playing` apresenta um peso japones relativamente maior que o dos demais generos lideres.

## Conclusoes finais

- A base esta bem completa, com ausencias concentradas em `Year` e `Publisher`.
- O auge do mercado no dataset aparece entre 2007 e 2010, com pico de vendas em 2008 e pico de lancamentos em 2009.
- `Action` lidera entre os generos, `PS2` lidera entre as plataformas e `Nintendo` lidera entre as publicadoras.
- `Wii Sports` e o maior destaque individual de vendas.
- A America do Norte tem o maior peso regional, mas o Japao aparece com mais forca relativa em `Role-Playing`.

Com isso, o notebook responde 9 perguntas com 9 graficos e deixa um roteiro pronto para apresentacao ou expansao da analise.